# CLIP 제로샷 분류 — Dogs vs Cats (PyTorch + transformers) — Colab

**CLIP**(Contrastive Language-Image Pre-training)으로 [Dogs vs Cats](https://www.kaggle.com/c/dogs-vs-cats-redux-kernels-edition) 이미지를 **학습 없이(zero-shot)** 분류하는 기본 예제입니다.

- CLIP 핵심: 이미지와 텍스트를 같은 공간에 임베딩 → **코사인 유사도**로 매칭.
- `"a photo of a cat"`, `"a photo of a dog"` 두 텍스트와 이미지의 유사도를 비교해 분류합니다. **모델 학습이 전혀 없습니다.**
- 실행 시 **Kaggle API로 데이터를 직접 다운로드**하며, 토큰이 **노트북에 내장**되어 바로 실행됩니다.
- 마지막에 제출 파일 `submission.csv` (`id,label` = 개일 확률) 가 `/content` 에 생성됩니다.

> ⚙️ **GPU 권장**: 런타임 → 런타임 유형 변경 → GPU (12,500장 추론, CPU는 느림).  
> ⚠️ **사전 필수**: `jinusun` 계정으로 [대회 규칙](https://www.kaggle.com/c/dogs-vs-cats-redux-kernels-edition/rules) Join 필요.  
> ⚠️ 노트북에 개인 API 토큰이 평문으로 들어 있습니다. 외부에 공유/업로드하지 마세요.

## 0. Setup — 라이브러리 설치 & Kaggle 자격증명

In [3]:
import sys, subprocess
for pkg in ["kaggle", "torch", "transformers", "numpy", "pandas", "pillow"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("라이브러리 준비 완료")

라이브러리 준비 완료


In [4]:
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"
print("Kaggle 자격증명 설정 완료 (내장 토큰)")

Kaggle 자격증명 설정 완료 (내장 토큰)


## 1. Kaggle에서 Dogs vs Cats 데이터 다운로드

> 약 800MB라 수 분 걸릴 수 있습니다.

In [5]:
import os, glob, zipfile
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()

api.competition_download_files("dogs-vs-cats-redux-kernels-edition", path=WORK_DIR, quiet=False)
for zp in glob.glob(os.path.join(WORK_DIR, "*.zip")):
    with zipfile.ZipFile(zp) as zf: zf.extractall(WORK_DIR)
    os.remove(zp)
for fn in ["train.zip", "test.zip"]:
    zp = os.path.join(WORK_DIR, fn)
    if os.path.exists(zp):
        with zipfile.ZipFile(zp) as zf: zf.extractall(WORK_DIR)
        os.remove(zp)

TRAIN_DIR = os.path.join(WORK_DIR, "train")
TEST_DIR = os.path.join(WORK_DIR, "test")
print("train images:", len(os.listdir(TRAIN_DIR)), "| test images:", len(os.listdir(TEST_DIR)))

100%|██████████| 814M/814M [00:48<00:00, 17.6MB/s]



train images: 25000 | test images: 12500


## 2. CLIP 제로샷 분류 & 제출 파일 생성
HuggingFace `transformers` 의 `pipeline('zero-shot-image-classification', model='openai/clip-vit-base-patch32')` 로 test 이미지를 **'고양이/개' 프롬프트**로 **제로샷 분류**(학습 없음) → **'개' 확률**을 `submission.csv`(`id, label`)로 저장합니다.

In [7]:
import os, glob
import pandas as pd
import numpy as np
import torch
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1
classifier = pipeline(
    "zero-shot-image-classification",
    model="openai/clip-vit-base-patch32",
    device=device
)
test_files = sorted(glob.glob(os.path.join(TEST_DIR, "*.jpg")),
                    key=lambda p: int(os.path.splitext(os.path.basename(p))[0]))
test_ids = [int(os.path.splitext(os.path.basename(p))[0]) for p in test_files]
candidate_labels = ["a photo of a cat", "a photo of a dog"]
results = classifier(test_files, candidate_labels=candidate_labels, batch_size=64)

test_prob = [
    next(item['score'] for item in res if item['label'] == "a photo of a dog")
    for res in results
]
submission_path = os.path.join(WORK_DIR, "submission.csv")
sub = pd.DataFrame({"id": test_ids, "label": np.clip(test_prob, 1e-4, 1 - 1e-4)})
sub.to_csv(submission_path, index=False)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

## 3. 제출 파일 내려받기

In [8]:
try:
    from google.colab import files
    files.download(submission_path)
except Exception as e:
    print("자동 다운로드 불가:", e)
    print("submission.csv 위치:", submission_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 🏁 제출하기

위에서 생성된 `submission.csv` 를 아래 페이지에서 업로드해 제출하세요:

👉 **[Dogs vs Cats Redux 제출 페이지](https://www.kaggle.com/c/dogs-vs-cats-redux-kernels-edition/submit)**

- 대회 홈: https://www.kaggle.com/c/dogs-vs-cats-redux-kernels-edition
> 제출하려면 해당 대회에 Join(규칙 동의)되어 있어야 합니다.